# 02 — Transformer (ModernBERT, o BERT per confronto)

Stessi fold e stessi articoli del classificatore a feature. Due input:

- **A** — le prime `PAROLE_ART` parole dell'articolo;
- **C** — lo stesso troncamento dell'articolo + le prime `PAROLE_PAG` parole delle pagine del sito.

Impostazioni vicine al paper: 5 epoche, batch 32 (eval 16), learning rate 2e-5, AdamW, precisione mista.
Con `EARLY_STOPPING` si tiene da parte il 15% dei **gruppi** di training come validation e si tiene l'epoca
con la validation loss più bassa (il test non viene mai guardato).

Salva tutto, e ogni fold finito viene saltato se il notebook è rilanciato:

- `Results/2-classificazione/<nome>/<config>_<schema>_fold<k>.csv` — `p_l1..p_l5`, `y_pred`, `y_true`, `domain`
- `Results/2-classificazione/<nome>_<config>_<schema>_articoli.csv` — l'unione dei fold
- `Results/2-classificazione/<nome>_loss.csv` — loss (e validation loss) per epoca, per ogni fold
- log: `Results/2-classificazione/avanzamento_<nome>.log`

`<nome>` è `modernbert`, `bert`, … con `_es` se c'è l'early stopping.

**Sul server** si lancia con papermill dentro `screen`, cambiando i parametri con `-p`:

```bash
cd ~/John/WebScience2027/EsperimentiPreliminari/2-NuovoScrapingConPagine/2-classificazione
screen -dmS modernbert_es papermill 02_transformer.ipynb ../Results/2-classificazione/esecuzioni/02_modernbert_es.ipynb -p EARLY_STOPPING True
```

In [ ]:
import gc
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, registratore, taglia

In [ ]:
MODELLO = "answerdotai/ModernBERT-base"   # oppure "bert-base-uncased" con MAX_TOKEN = 256
MAX_TOKEN = 1024
PAROLE_ART = 400           # ~512 token di articolo
PAROLE_PAG = 400           # ~512 token di pagine; l'articolo è identico in A e C, C aggiunge le pagine
CONFIG = ["A", "C"]
SCHEMI_DA_FARE = SCHEMI
EPOCHE = 5
FOLDS = None               # quali fold (None = tutti)
EARLY_STOPPING = False
PATIENCE = 1
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [ ]:
def fissa_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


nome = "modernbert" if "ModernBERT" in MODELLO else ("bert" if "bert" in MODELLO else MODELLO.split("/")[-1])
if EARLY_STOPPING:
    nome += "_es"
OUT = cartella_risultati(PROVA)
per_fold = OUT / nome
per_fold.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT / f"avanzamento_{nome}.log")
loss_file = OUT / f"{nome}_loss.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
registra(f"=== {MODELLO} (max {MAX_TOKEN} token) su {device} ===")
art = carica(PROVA)
pag_cut = {d: taglia(t, PAROLE_PAG) for d, t in art.drop_duplicates("domain")[["domain", "pagine"]].values}
art["A"] = art.testo.map(lambda t: taglia(t, PAROLE_ART))   # solo articolo (stesso troncamento in A e C)
art["C"] = art.A + "\n\n" + art.domain.map(pag_cut)        # articolo + pagine
tok = AutoTokenizer.from_pretrained(MODELLO)

## Un fold

In [ ]:
def tensori(sotto, config, con_y=True):
    enc = tok(sotto[config].tolist(), truncation=True, padding="max_length",
              max_length=MAX_TOKEN, return_tensors="pt")
    if con_y:
        return TensorDataset(enc.input_ids, enc.attention_mask, torch.tensor(sotto.label.tolist()))
    return TensorDataset(enc.input_ids, enc.attention_mask)


def val_loss(model, dl_val):
    model.eval(); tot = 0.0
    with torch.no_grad(), torch.cuda.amp.autocast():
        for ids, mask, y in dl_val:
            tot += model(ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss.item()
    return tot / len(dl_val)


def un_fold(config, schema, fold):
    """Addestra sul training del fold e restituisce le probabilità sul test e la loss per epoca."""
    fissa_seed(SEED)
    tr_all, te = art[art[schema] != fold], art[art[schema] == fold]
    # early stopping: tengo da parte il 15% dei GRUPPI di training come validation
    dl_val = None
    if EARLY_STOPPING:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
        i_tr, i_val = next(gss.split(tr_all, tr_all.label, tr_all.gruppo))
        tr, val = tr_all.iloc[i_tr], tr_all.iloc[i_val]
        dl_val = DataLoader(tensori(val, config), batch_size=16)
    else:
        tr = tr_all
    registra(f"{config} {schema} fold {fold}: inizio ({len(tr)} training"
             + (f", {len(val)} validation" if EARLY_STOPPING else "") + f", {len(te)} test)")
    dl_tr = DataLoader(tensori(tr, config), batch_size=32, shuffle=True)
    dl_te = DataLoader(tensori(te, config, con_y=False), batch_size=16)
    kw = {"reference_compile": False} if "ModernBERT" in MODELLO else {}   # niente torch.compile: tempi stabili
    model = AutoModelForSequenceClassification.from_pretrained(MODELLO, num_labels=5, **kw).to(device)
    opt = AdamW(model.parameters(), lr=2e-5)
    scaler = torch.cuda.amp.GradScaler()                 # precisione mista: addestramento più veloce
    miglior_val, miglior_stato, attese, righe_loss = float("inf"), None, 0, []
    for epoca in range(1, EPOCHE + 1):
        model.train(); t0, tot = time.time(), 0.0
        for ids, mask, y in dl_tr:
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = model(ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        media = tot / len(dl_tr)
        vl = val_loss(model, dl_val) if EARLY_STOPPING else None
        righe_loss.append({"config": config, "schema": schema, "fold": fold, "epoca": epoca,
                           "loss": media, "val_loss": vl})
        registra(f"{config} {schema} fold {fold}: epoca {epoca}/{EPOCHE} in {time.time()-t0:.0f} s, "
                 f"loss {media:.4f}" + (f", val_loss {vl:.4f}" if EARLY_STOPPING else ""))
        if EARLY_STOPPING:
            if vl < miglior_val:
                miglior_val, miglior_stato, attese = vl, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
            else:
                attese += 1
                if attese > PATIENCE:
                    registra(f"{config} {schema} fold {fold}: early stop all'epoca {epoca} (migliore: val_loss {miglior_val:.4f})")
                    break
    if EARLY_STOPPING and miglior_stato is not None:
        model.load_state_dict(miglior_stato)
    model.eval(); probs = []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for ids, mask in dl_te:
            logit = model(ids.to(device), attention_mask=mask.to(device)).logits
            probs.append(torch.softmax(logit.float(), dim=1).cpu().numpy())
    P = np.vstack(probs)
    out = pd.DataFrame(P, columns=CLASSI)
    out.insert(0, "domain", te.domain.values); out.insert(0, "article_id", te.article_id.values)
    out["y_true"] = te.label.values; out["y_pred"] = P.argmax(1)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    registra(f"{config} {schema} fold {fold}: fatto, acc articoli {(out.y_true == out.y_pred).mean():.3f}")
    return out, righe_loss

## Tutti i fold

La loss di ogni fold viene accodata a `<nome>_loss.csv` appena il fold finisce.

In [ ]:
for config in CONFIG:
    for schema in SCHEMI_DA_FARE:
        parti = []
        folds = FOLDS if FOLDS else sorted(art[schema].unique())
        for fold in folds:
            ff = per_fold / f"{config}_{schema}_fold{fold}.csv"
            if ff.exists():
                registra(f"{config} {schema} fold {fold}: già fatto, salto")
            else:
                out, righe_loss = un_fold(config, schema, fold)
                out.to_csv(ff, index=False)
                pd.DataFrame(righe_loss).to_csv(loss_file, mode="a", header=not loss_file.exists(), index=False)
            parti.append(pd.read_csv(ff))
        pd.concat(parti).to_csv(OUT / f"{nome}_{config}_{schema}_articoli.csv", index=False)
        registra(f"=== {config} {schema}: tutti i fold pronti ===")
registra("=== FINITO ===")

## Risultati

Le stesse letture del classificatore a feature, e la loss per epoca (tratteggio: validation, se c'è).

In [ ]:
pred = {}
for config in CONFIG:
    for schema in SCHEMI_DA_FARE:
        f = OUT / f"{nome}_{config}_{schema}_articoli.csv"
        if f.exists():
            pred[(config, schema)] = pd.read_csv(f)
ris = pd.DataFrame([{"config": c, "fold": NOME_SCHEMA[s], **letture(p)} for (c, s), p in pred.items()])
ris = ris.set_index(["config", "fold"])
ris.to_csv(OUT / f"{nome}_metriche.csv")
ris

In [ ]:
if loss_file.exists():
    loss = pd.read_csv(loss_file).drop_duplicates(["config", "schema", "fold", "epoca"], keep="last")
    fig, assi = plt.subplots(1, len(CONFIG), figsize=(5 * len(CONFIG), 3.5), sharey=True, squeeze=False)
    for ax, config in zip(assi[0], CONFIG):
        for (schema, fold), g in loss[loss.config == config].groupby(["schema", "fold"]):
            colore = "C0" if schema == "fold_gruppo" else "C1"
            ax.plot(g.epoca, g.loss, color=colore, alpha=0.6)
            if g.val_loss.notna().any():
                ax.plot(g.epoca, g.val_loss, "--", color=colore, alpha=0.6)
        ax.set_title(f"{nome}, input {config} (blu: per gruppo, arancio: per testata)", fontsize=9)
        ax.set_xlabel("epoca"); ax.set_yscale("log")
    assi[0][0].set_ylabel("loss")
    plt.tight_layout(); plt.show()